# Stretch: more git, worked solutions

The answers to the stretch notebook. Try each one before looking.

Everything here is terminal work, so this notebook has no code to run.
The commands are shown with the output they give, trimmed to the lines
that matter. Your commit hashes (the short codes like `a82e6c1`) will be
different from these; that is expected.

On Windows you may also see `warning: ... LF will be replaced by CRLF`.
It is about line endings, it is harmless, and you can ignore it.

## The playground

```bash
cd ..
mkdir git-playground
cd git-playground
git init
```

```
Initialized empty Git repository in .../git-playground/.git/
```

Why not inside the course folder? Because git would see the whole
playground as one untracked folder in your course repo, and a repo nested
inside another is a classic source of confusion. Next to it, the two never
meet.

## Stretch 1: read the history like a detective

```bash
git log --oneline --author="Your Name"
```

Use the name you gave `git config user.name`. It matches part of the name,
so your first name alone usually works. Everything else in the log came
with the course when you cloned it.

```bash
git log --oneline -- sessions/session-04/notebooks/03-explore-plays-exercise.ipynb
```

Only the commits that changed that one file. The `--` means "what follows
is a file path, not a branch name". Use any file of yours.

```bash
git log --stat -3
```

```
commit 4270989...
    Add Wish You Were Here

 songs.txt | 2 ++
 1 file changed, 2 insertions(+)
```

`--stat` is the summary: which files, and how many lines added (`+`) and
removed (`-`). Add `--oneline` for a more compact version. Swap `--stat`
for `-p` and you get the full diff of every commit instead.

```bash
git log --oneline --grep="session 4"
```

Commits whose message contains those words. This is where good commit
messages pay off: you can only find what you described.

```bash
git log --oneline --graph --all -15
```

The left side is a drawing of the history. Each `*` is a commit, and the
lines show which commit came before which. Where a line splits off, a
branch was made; where two lines join, they were merged. `--all` includes
every branch, not only the one you are on. In this repo you can see the
course's own merged pull requests.

## Stretch 2: fix the last commit

```bash
git add songs.txt
git commit -m "Add Echos"
git log --oneline
```

```
a82e6c1 Add Echos
e857507 Add a list of songs
```

```bash
git commit --amend -m "Add Echoes"
git log --oneline
```

```
edb60b9 Add Echoes
e857507 Add a list of songs
```

Still two commits, but the hash changed from `a82e6c1` to `edb60b9`.
Amend does not edit a commit. It makes a **new** one and quietly puts it
in place of the old.

**Harder:**

```bash
git add songs.txt
git commit -m "Add Wish You Were Here"
# now add Hotel California to the file and save
git add songs.txt
git commit --amend --no-edit
git show --stat HEAD
```

```
    Add Wish You Were Here

 songs.txt | 2 ++
 1 file changed, 2 insertions(+)
```

Two lines in one commit, same message. `--no-edit` means "keep the
message as it is". Without it, git would open an editor to ask.

**Why not amend a pushed commit?** Because the hash changes. GitHub has
the old commit and your laptop has the new one in its place, so as far as
git is concerned the two copies now have **different histories**. Try to
push and you get:

```
Your branch and 'origin/main' have diverged
...
 ! [rejected]        main -> main (non-fast-forward)
```

The way people "fix" that online is `git push --force`, which overwrites
GitHub's history with yours. On your own repo that is survivable; on a
repo other people use, you have just deleted a commit they may have built
on. So the rule is simple: **amend freely until you push, never after.**
Once it is pushed, a typo in a message is something you live with, and a
mistake in the code gets a new commit that fixes it.

## Stretch 3: `.gitignore`, and the trap in it

Before the `.gitignore`:

```
Untracked files:
	private/
	scratch.txt
```

A `.gitignore` with two lines:

```
scratch.txt
private/
```

The trailing `/` means "this folder and everything in it". Now `git status`
only shows the new `.gitignore` itself, which you **do** want to commit,
because the rules should travel with the repo.

```bash
git check-ignore -v private/secret.txt
```

```
.gitignore:2:private/	private/secret.txt
```

File `.gitignore`, line 2, the rule `private/`. When a repo has several
`.gitignore` files and a rule you have forgotten about, this is how you
find the culprit. On `songs.txt` it prints **nothing at all**: no output
means "not ignored".

```bash
git add .gitignore
git commit -m "Ignore scratch files"
```

**The trap.** After committing `debug.log` and then adding `*.log` to
`.gitignore`:

```
Changes not staged for commit:
	modified:   .gitignore
	modified:   debug.log
```

Git is not ignoring it. **`.gitignore` only affects files git is not
already tracking.** Once a file has been committed, git keeps watching it,
whatever the ignore rules say.

```bash
git rm --cached debug.log
git status
```

```
Changes to be committed:
	deleted:    debug.log
```

That "deleted" looks alarming, but it means "deleted from git's list of
tracked files". `--cached` means "only in git, leave my folder alone", and
the file is still there in VS Code.

```bash
git add .gitignore
git commit -m "Stop tracking debug.log"
git status
```

```
nothing to commit, working tree clean
```

This is exactly the situation with a password file committed by mistake,
and it is why session 5 says a pushed secret is public forever: `git rm
--cached` stops tracking it **from now on**, but every earlier commit
still contains it.

## Stretch 4: put work aside with `git stash`

With an uncommitted line in `songs.txt`:

```bash
git switch experiment
```

```
error: Your local changes to the following files would be overwritten by checkout:
	songs.txt
Please commit your changes or stash them before you switch branches.
Aborting
```

Git refuses because switching would replace `songs.txt` with the
`experiment` version and your unsaved line would be lost. The message
names both ways out: commit, or stash.

```bash
git stash
git stash list
```

```
Saved working directory and index state WIP on main: 51ca2c9 Stop tracking debug.log
stash@{0}: WIP on main: 51ca2c9 Stop tracking debug.log
```

`WIP` means "work in progress". `git status` is clean now, so switching is
allowed:

```bash
git switch experiment     # Stairway to Heaven is at the end of the file
git switch main
git stash pop
```

```
	modified:   songs.txt
...
Dropped refs/stash@{0} (...)
```

The half-finished line is back, and `pop` removed it from the shelf, so
`git stash list` is empty again. Then `git restore songs.txt` throws it
away.

A stash is meant for minutes, not weeks. It has no message worth reading
and it is easy to forget it exists. If the work matters, commit it on a
branch instead.

## Stretch 5: undo a commit with `git revert`

```bash
git log --oneline -3
```

```
3f7c33e Add a README
f90223a Add Baby Shark
51ca2c9 Stop tracking debug.log
```

```bash
git revert --no-edit f90223a
git log --oneline -4
```

```
1d7589d Revert "Add Baby Shark"
3f7c33e Add a README
f90223a Add Baby Shark
51ca2c9 Stop tracking debug.log
```

Baby Shark is gone from `songs.txt`, and the README is still there. But
the Baby Shark commit is **still in the history**. Revert did not remove
it; it added a new commit that does the exact opposite. `--no-edit`
accepts git's suggested message, `Revert "Add Baby Shark"`, instead of
opening an editor.

This went smoothly because nothing after Baby Shark touched the lines
next to it. If a later commit had changed the line right below, git would
stop with a conflict, which is stretch 6.

**Why revert and not `reset --hard`?** Three reasons, in order of
importance:

1. **It does not rewrite history.** Like amend, `reset` changes what the
   history *is*. If the commit was already pushed, GitHub and your laptop
   now disagree, and you are back to the force-push problem from stretch
   2. A revert is just one more commit, so a normal `git push` works.
2. **It is honest.** Anybody reading the log, including you in six
   months, can see that Baby Shark was added and then deliberately taken
   out, and why.
3. **It is itself undoable.** A revert is a commit, so you can revert the
   revert. `reset --hard` also throws away any **uncommitted** changes in
   your folder, and those are not in git at all, so nothing can bring
   them back.

`reset` has real uses, but you do not need it yet. `restore` for
uncommitted changes and `revert` for committed ones cover everything.

## Stretch 6: a merge conflict, on purpose

```bash
git merge new-title
```

```
Auto-merging songs.txt
CONFLICT (content): Merge conflict in songs.txt
Automatic merge failed; fix conflicts and then commit the result.
```

**1.** `git status` explains where you are and how to get out:

```
You have unmerged paths.
  (fix conflicts and run "git commit")
  (use "git merge --abort" to abort the merge)

Unmerged paths:
	both modified:   songs.txt
```

**2.** The file:

```
<<<<<<< HEAD
My favourite songs
=======
Songs I love
>>>>>>> new-title

Hey Jude
...
```

Between `<<<<<<< HEAD` and `=======` is your version, from the branch you
are on (`main`). Between `=======` and `>>>>>>> new-title` is the version
from the branch you are merging in. Everything outside the markers merged
without trouble; only the one line is in dispute.

**3.** `git merge --abort` puts everything back exactly as it was before
the merge, and `git status` is clean. Remember this one. Whenever a merge
goes somewhere you do not understand, it is the way back.

**4.** Resolving means making the file look the way you want and removing
the three marker lines, for example:

```
My favourite songs

Hey Jude
...
```

Git does not check what you chose. It only checks that you marked the
file as done with `git add`, so read the file once before you do.

**5.**

```bash
git add songs.txt
git commit --no-edit
git log --oneline --graph
```

```
*   259b472 Merge branch 'new-title'
|\
| * 6f63548 Rename the list to Songs I love
* | c4def77 Rename the list to My favourite songs
|/
* 1d7589d Revert "Add Baby Shark"
```

Two lines of work splitting apart and joining again, which is what the
graph in stretch 1 was drawing.

**Why can git not just pick one?** Git merges *lines*, not meaning. When
the two branches changed different lines, it can safely keep both. When
they changed the **same** line differently, both versions are equally
"the latest", and choosing between them is a decision about what the file
should say. Only a person knows that.

If git guessed, it would sometimes guess wrong **silently**, and you
would find out much later that a change had vanished. Stopping and asking
is annoying, but it is the safe behaviour. A conflict is not an error; it
is git refusing to make a decision that is yours.

## Stretch 7: a pull request on your own repo

```bash
git switch -c add-cheatsheet
# write GIT-CHEATSHEET.md
git add GIT-CHEATSHEET.md
git commit -m "Add a git cheatsheet in my own words"
git push -u origin add-cheatsheet
```

```
 * [new branch]      add-cheatsheet -> add-cheatsheet
branch 'add-cheatsheet' set up to track 'origin/add-cheatsheet'.
```

The branch now exists on GitHub too, and `main` there is untouched. That
is the whole idea of a pull request: the work sits next to `main`,
visible, until somebody decides to merge it.

On GitHub: **Compare & pull request**, a title, **Create pull request**,
then the **Files changed** tab. That tab is the same diff as `git diff`,
but it is where reviewers comment line by line. Even on a solo repo it is
a good habit to read your own change there once before merging.

Keep "Create a merge commit" when merging. The other two options
(squash, rebase) rewrite the commits, and then `git branch -d` on your
laptop will not recognise the branch as merged.

Back in the terminal:

```bash
git switch main
git pull
git branch -d add-cheatsheet
git log --oneline --graph -5
```

```
Fast-forward
 GIT-CHEATSHEET.md | 1 +
...
Deleted branch add-cheatsheet (was d5577b5).

*   0bde1d5 Merge pull request #1 from YOUR-NAME/add-cheatsheet
|\
| * d5577b5 Add a git cheatsheet in my own words
|/
* fccf4e1 ...
```

The merge commit was made on GitHub, and `git pull` brought it down. The
deleted branch may still show up in `git branch -a` as
`remotes/origin/add-cheatsheet`. `git fetch --prune` tidies that away.

This is the same workflow as merging a branch in the terminal, with one
difference that matters at work: the merge happens where other people
can see and discuss it first.